# 04 · Agentic RAG with LangGraph
The full agent: **route → rewrite → retrieve → grade → (refine loop) → generate → verify → finalize**, with bounded retries and a fail-closed `no_answer` path.

In [1]:
# ---- Shared setup (identical in every notebook) -------------------------------
import os, sys, json, pathlib, time
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "sample_data"))

# OFFLINE=True  -> deterministic hash embeddings + mock LLM + local Chroma. No keys, no network.
# OFFLINE=False -> REAL Gemini/Groq/OpenAI + Chroma Cloud from your .env  (set NB_OFFLINE=0 or edit below)
OFFLINE = os.getenv("NB_OFFLINE", "1") == "1"
if OFFLINE:
    os.environ["RAG_OFFLINE"] = "1"
    os.environ["CHROMA_LOCAL_DIR"] = str(ROOT / "chroma_notebook")   # shared by all notebooks
else:
    os.environ["RAG_OFFLINE"] = "0"

from app.config import get_settings
S = get_settings()
print("mode            :", "OFFLINE (mock)" if S.offline else "LIVE providers")
print("vector store    :", "Chroma Cloud" if S.use_chroma_cloud else f"local ({S.chroma_local_dir})")
print("embedding       :", "hash512" if S.offline else S.embedding_provider)
print("LLM providers   :", [p for p, k in (("gemini", S.gemini_key), ("groq", S.groq_key), ("openai", S.openai_key)) if k] or "none (offline mock)")

mode            : OFFLINE (mock)
vector store    : local (/home/claude/proj/v1/rag_service/chroma_notebook)
embedding       : hash512
LLM providers   : ['gemini', 'groq', 'openai']


## Graph structure

In [2]:
from app.agent import get_graph
print(get_graph().get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	route(route)
	rewrite(rewrite)
	retrieve(retrieve)
	grade(grade)
	refine(refine)
	generate(generate)
	verify(verify)
	no_answer(no_answer)
	finalize(finalize)
	__end__([<p>__end__</p>]):::last
	__start__ --> route;
	generate -.-> no_answer;
	generate -.-> verify;
	grade -.-> generate;
	grade -.-> no_answer;
	grade -.-> refine;
	no_answer --> finalize;
	refine --> retrieve;
	retrieve --> grade;
	rewrite --> retrieve;
	route -.-> finalize;
	route -.-> rewrite;
	verify -.-> finalize;
	verify -.-> generate;
	verify -.-> no_answer;
	finalize --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



Paste the Mermaid text into https://mermaid.live to see the diagram.

In [3]:
from app.agent import ask
from app.store import get_store
assert get_store().count() > 0, "Run notebook 01 first"
def run(q, subj="BIO101", history=None):
    out = ask(q, subject_codes=[subj], history=history or [])
    print("Q:", q); print("A:", out["answer"])
    print(f"   confidence={out['confidence']} grounded={out['grounded']} intent={out['intent']} latency={out['latency_ms']}ms")
    print("   sources:", [(s['file_name'], 'p.%s' % s['page']) for s in out["sources"]])
    print("   path   :", " → ".join(t["node"] for t in out["agent_trace"]), "\n")
    return out
run("hello")
run("Which organelle produces ATP?")
run("Where does photosynthesis take place in a plant cell?")
run("Who won the football world cup in 1998?")
run("what is the weather today")

Q: hello
A: Hi! I'm your course assistant. Ask me anything about the material your faculty has uploaded.
   confidence=1.0 grounded=True intent=smalltalk latency=2ms
   sources: []
   path   : route → finalize 

Q: Which organelle produces ATP?
A: Mitochondria are the organelles that produce most of the cell's ATP through cellular respiration. [1]
   confidence=1.0 grounded=True intent=academic latency=7ms
   sources: [('bio_notes.pdf', 'p.1')]
   path   : route → rewrite → retrieve → grade → generate → verify → finalize 

Q: Where does photosynthesis take place in a plant cell?
A: Cell Biology Notes
Photosynthesis occurs in the chloroplasts of plant cells. [1]
   confidence=0.982 grounded=True intent=academic latency=5ms
   sources: [('bio_notes.pdf', 'p.1')]
   path   : route → rewrite → retrieve → grade → generate → verify → finalize 

Q: Who won the football world cup in 1998?
A: I couldn't find this in the course material that has been uploaded for your subject, so I won't guess. 

{'answer': 'I can only help with questions about your course material. Try asking about a topic from your subject.',
 'confidence': 1.0,
 'grounded': True,
 'intent': 'offtopic',
 'sources': [],
 'retrieval': {'strategy': 'hybrid_rrf'},
 'agent_trace': [{'node': 'route', 't': 1791512321.2, 'intent': 'offtopic'},
  {'node': 'finalize', 't': 1791512321.2}],
 'standalone_question': None,
 'latency_ms': 1}

## Multi-turn: follow-up question resolved with chat history

In [4]:
h = [{"role":"user","content":"What is DNA replication?"},{"role":"assistant","content":"It copies DNA before cell division."}]
o = run("Which enzyme does it use?", history=h)
print("rewritten:", o["standalone_question"])

Q: Which enzyme does it use?
A: The enzyme DNA polymerase adds nucleotides in the 5' to 3' direction. [1]
   confidence=0.861 grounded=True intent=academic latency=5ms
   sources: [('bio_notes.pdf', 'p.2')]
   path   : route → rewrite → retrieve → grade → generate → verify → finalize 

rewritten: Which enzyme does it use?


## Scope enforcement: same question, other subject → refusal

In [5]:
run("Which organelle produces ATP?", subj="PHY101")

Q: Which organelle produces ATP?
A: I couldn't find this in the course material that has been uploaded for your subject, so I won't guess. I've flagged it so your faculty can add or clarify the material.
   confidence=0.1 grounded=False intent=academic latency=7ms
   sources: []
   path   : route → rewrite → retrieve → grade → refine → retrieve → grade → no_answer → finalize 



{'answer': "I couldn't find this in the course material that has been uploaded for your subject, so I won't guess. I've flagged it so your faculty can add or clarify the material.",
 'confidence': 0.1,
 'grounded': False,
 'intent': 'academic',
 'sources': [],
 'retrieval': {'strategy': 'hybrid_rrf',
  'dense_hits': 3,
  'sparse_hits': 0,
  'candidates': 3},
 'agent_trace': [{'node': 'route', 't': 1791512321.221, 'intent': 'academic'},
  {'node': 'rewrite',
   't': 1791512321.222,
   'queries': ['Which organelle produces ATP?']},
  {'node': 'retrieve',
   't': 1791512321.224,
   'round': 0,
   'candidates': 3,
   'dense_hits': 3,
   'sparse_hits': 0},
  {'node': 'grade', 't': 1791512321.225, 'kept': 0, 'of': 3},
  {'node': 'refine', 't': 1791512321.225, 'queries': ['organell produc atp']},
  {'node': 'retrieve',
   't': 1791512321.227,
   'round': 1,
   'candidates': 3,
   'dense_hits': 3,
   'sparse_hits': 0},
  {'node': 'grade', 't': 1791512321.227, 'kept': 0, 'of': 3},
  {'node': 'n

## Inspect the full trace of one run

In [6]:
import pandas as pd
pd.DataFrame(ask("Which enzyme adds nucleotides during DNA replication?", subject_codes=["BIO101"])["agent_trace"]).drop(columns="t")

,node,intent,queries,round,candidates,dense_hits,sparse_hits,kept,of,attempt,chars,support,verdict,confidence,cited
0,route,academic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,rewrite,NaN,[Which enzyme adds nucleotides during DNA repl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,retrieve,NaN,NaN,0.0,3.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,grade,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN
4,generate,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,73.0,NaN,NaN,NaN,NaN
5,verify,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,grounded,NaN,NaN
6,finalize,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.917,[1]
